## Medium Articles Analyzer
This agent will read articles, figure out what they're about, extract important elements, and deliver clean summaries — essentially your personal research assistant.

In [ ]:
from typing_extensions import TypedDict
from typing import List

from langgraph.graph import StateGraph, END
from langchain_core.messages import HumanMessage

In [27]:

import os
from dotenv import load_dotenv
import litellm

load_dotenv()

API_BASE = os.getenv("OLLAMA_API_BASE", "http://127.0.0.1:11434")
MODEL_ID = os.getenv("OLLAMA_MODEL_ID", "ollama/deepseek-r1:latest")
# Function to call your local LLM using litellm + Ollama
def call_ollama(messages):
    response = litellm.completion(
        model=MODEL_ID,
        api_base=API_BASE,
        messages=messages,
        custom_llm_provider="ollama"   # <--- This tells litellm to use Ollama as the backend
    )
    return response["choices"][0]["message"]["content"].strip()


In [28]:

# Defining the State Schema

class State(TypedDict): # TypeDictprovides type safety, warning us if we store incorrect data types
    text: str                    # Original input text
    classification: str         # Output of classification node
    entities: List[str]         # Output of entity extraction node
    summary: str                # Output of summarization node

# Step:  Define Graph Nodes (Tools)
# ---- Node 1: Classification ----

# Adding Agent's Capabilities 
# Now we'll build specialized tools for our agent, each handling a specific task type.


def classification_node(state: State) -> dict:
    """
    Classification node: classifies the input text into categories : News, Blog, Research, or Other.

    Parameters:
    - state: dict-like object holding the current text and outputs

    Returns:
    - Updated state with a new key 'classification'
    """
    # Construct the prompt for classification
    prompt = [
        {"role": "system", "content": "You are a helpful assistant that classifies text into News, Blog, Research, or Other."},
        {"role": "user", "content": f"Classify the following text into one of the categories:\n\n{state['text']}"}
    ]
    # Call the Ollama model with the prompt
    classification = call_ollama(prompt)

    # Store the classification result in the state dictionary
    state["classification"] = classification
    return state


In [29]:
# ---- Node 2: Entity Extraction ----
# This function processes the document and returns a list of key entities like important names, organizations, and places.

def entity_extraction_node(state: State) -> dict:
    """
    Entity extraction node: extracts named entities from the input text  like Person, Organization, and Location

    Parameters:
    - state: dict-like object holding the current text and outputs

    Returns:
    - Updated state with a new key 'entities' (list of extracted entities)
    """
    prompt = [
        {"role": "system", "content": "You are a named entity recognizer that extracts Persons, Organizations, and Locations."},
        {"role": "user", "content": f"Extract entities (Person, Organization, Location) from the following text:\n\n{state['text']}"}
    ]

    # Get the entities string from the model
    entities_str = call_ollama(prompt)
    
    # Convert the comma-separated string into a clean list of entity names
    entities = [e.strip() for e in entities_str.split(",") if e.strip()]
                
    # Save entities in the state
    state["entities"] = entities
    return state

In [30]:
# ---- Node 3: Summarization ----
# This function distills the document into a concise summary of its main points.
def summarization_node(state: State) -> dict:
    """
    Summarization node: generates a short summary of the input text

    Parameters:
    - state: dict-like object holding the current text and outputs

    Returns:
    - Updated state with a new key 'summary'
    """
    prompt = [
        {"role": "system", "content": "You summarize text concisely."},
        {"role": "user", "content": f"Summarize the following text in one short sentence:\n\n{state['text']}"}
    ]

    # Call the Ollama model for summary
    summary = call_ollama(prompt)

    # Save summary in the state
    state["summary"] = summary
    return state


In [31]:
# ---- Build the LangGraph Workflow ----
# Now we'll connect these capabilities into a coordinated workflow
workflow = StateGraph(State)

# Add nodes to the workflow graph
workflow.add_node("classify", classification_node)
workflow.add_node("extract_entities", entity_extraction_node)
workflow.add_node("summarize", summarization_node)

# Define the execution flow:
# Start from classification → then extract entities → then summarize → then END
workflow.set_entry_point("classify")
workflow.add_edge("classify", "extract_entities")
workflow.add_edge("extract_entities", "summarize")
workflow.add_edge("summarize", END)

# Compile the workflow into an executable app
app = workflow.compile()

# ---- Test Run ----
sample_text = """
Anthropic's MCP (Model Context Protocol) is an open-source powerhouse that lets your applications interact effortlessly with APIs across various systems.
"""

initial_state = {"text": sample_text}
result = app.invoke(initial_state)

# ---- Output ----
print("Classification:", result["classification"])
print("Entities:", result["entities"])
print("Summary:", result["summary"])

Classification: <think>
Okay, so I need to classify this text into one of four categories: News, Blog, Research, or Other. The user has provided a sentence about Anthropic's MCP.

First, let me read the text carefully. It says, "Anthropic's MCP (Model Context Protocol) is an open-source powerhouse that lets your applications interact effortlessly with APIs across various systems."

Looking at this, it's introducing MCP from Anthropic. They're explaining what MCP is and its purpose. MCP is described as an open-source tool that allows easy interaction with APIs on different systems.

Now, I need to figure out if this fits into News, Blog, Research, or Other. Let's break down the categories:

- **News**: This usually refers to recent events, announcements, or significant updates in a particular field.
- **Blog**: This is about articles that provide insights, reviews, or discussions on specific topics.
- **Research**: This involves detailed studies, analyses, or academic content related to

##  Limitations of AI Agents
- **Rigid Architecture:** Agents follow fixed node and graph structures — limited flexibility.  
- **Contextual Understanding:** Agents understand text but lack broader cultural or real-world knowledge without added data sources like internet search.  
- **Black Box Issue:** Internals of decision-making are mostly opaque, even if models like GPT or DeepSeek provide some reasoning traces.  
- **Human Oversight Needed:** AI agents are not fully autonomous; validation and supervision are crucial to ensure accuracy.

---

## Why This Matters

Understanding these limitations doesn’t undermine AI agents — it grounds expectations and improves system design, leading to solutions that work reliably and are complemented by human expertise.

---
